# Análise Exploratória de Negócio — Olist

## Objetivo

Analisar os dados do e-commerce brasileiro Olist utilizando
as tabelas da camada Gold.

A análise busca responder perguntas sobre:

- Evolução das vendas e faturamento
- Comportamento dos clientes
- Desempenho dos produtos e categorias
- Distribuição geográfica das vendas
- Eficiência das entregas
- Satisfação dos consumidores

Fonte: Brazilian E-Commerce Public Dataset by Olist.

Camada utilizada: workspace.gold.

####Qual é a distribuição dos pedidos por status e qual percentual foi efetivamente entregue?

In [0]:
%sql

SELECT
    order_status AS status_pedido,
    COUNT(*) AS quantidade_pedidos,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentual_pedidos

FROM workspace.gold.fato_pedidos

GROUP BY order_status

ORDER BY quantidade_pedidos DESC;

status_pedido,quantidade_pedidos,percentual_pedidos
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


### Insight 01 — Distribuição dos pedidos

A análise identificou 99.441 pedidos registrados no dataset Olist.

Desses pedidos, 96.478 (97,02%) apresentam status de entrega
concluída.

Os pedidos cancelados e indisponíveis representam aproximadamente
1,24% do total.

Esse resultado fornece uma visão inicial do comportamento operacional
dos pedidos e estabelece uma base para investigar atrasos, cancelamentos
e satisfação dos consumidores.

**Observação:** os percentuais representam os status registrados
no período histórico do dataset.

## 2. Evolução mensal do faturamento

**Pergunta de negócio:**

Como o faturamento bruto dos itens vendidos evoluiu ao longo do tempo?

**Objetivo:**

Identificar tendências de crescimento, sazonalidade e possíveis
variações no desempenho comercial do e-commerce.

In [0]:
%sql

SELECT
    t.ano_mes,

    COUNT(DISTINCT f.order_id) AS total_pedidos,

    COUNT(*) AS total_itens,

    ROUND(
        SUM(f.valor_produto),
        2
    ) AS faturamento_produtos,

    ROUND(
        SUM(f.valor_frete),
        2
    ) AS faturamento_frete,

    ROUND(
        SUM(f.valor_total_item),
        2
    ) AS faturamento_total

FROM workspace.gold.fato_itens_pedido f

INNER JOIN workspace.gold.dim_tempo t
    ON f.data_id = t.data_id

WHERE 
    t.ano_mes BETWEEN '2017-01' AND '2018-08'

GROUP BY
    t.ano_mes

ORDER BY
    t.ano_mes;

ano_mes,total_pedidos,total_itens,faturamento_produtos,faturamento_frete,faturamento_total
2017-01,789,955,120312.87,16875.62,137188.49
2017-02,1733,1951,247303.02,38977.60,286280.62
2017-03,2641,3000,374344.30,57704.29,432048.59
2017-04,2391,2684,359927.23,52495.01,412422.24
2017-05,3660,4136,506071.14,80119.81,586190.95
2017-06,3217,3583,433038.60,69924.44,502963.04
2017-07,3969,4519,498031.48,86940.14,584971.62
2017-08,4293,4910,573971.68,94232.92,668204.60
2017-09,4243,4831,624401.69,95997.22,720398.91
2017-10,4568,5322,664219.43,105092.94,769312.37


Databricks visualization. Run in Databricks to view.

#### Insight 02 — Evolução mensal do faturamento

A análise identificou crescimento expressivo do faturamento bruto
ao longo de 2017.

O mês de novembro de 2017 apresentou aproximadamente R$ 1,18 milhão
em faturamento, com 7.451 pedidos.

Esse comportamento sugere uma possível influência de eventos
sazonais, como a Black Friday.

Durante os primeiros meses de 2018, o faturamento permaneceu
em patamares elevados, superando R$ 1 milhão em diversos meses.

**Observação:** os valores representam o faturamento bruto
dos produtos acrescido do frete, sem dedução de custos,
cancelamentos ou devoluções.

### 3. Análise do ticket médio mensal

**Pergunta de negócio:**

Como o valor médio dos pedidos evoluiu ao longo do tempo?

**Objetivo:**

Investigar o comportamento do ticket médio e sua relação
com o crescimento do faturamento do e-commerce.

In [0]:
%sql

SELECT
    t.ano_mes,

    COUNT(DISTINCT f.order_id) AS total_pedidos,

    ROUND(
        SUM(f.valor_total_item),
        2
    ) AS faturamento_total,

    ROUND(
        SUM(f.valor_total_item)
        / COUNT(DISTINCT f.order_id),
        2
    ) AS ticket_medio

FROM workspace.gold.fato_itens_pedido f

INNER JOIN workspace.gold.dim_tempo t
    ON f.data_id = t.data_id

WHERE t.ano_mes BETWEEN '2017-01' AND '2018-08'

GROUP BY
    t.ano_mes

ORDER BY
    t.ano_mes;

ano_mes,total_pedidos,faturamento_total,ticket_medio
2017-01,789,137188.49,173.88
2017-02,1733,286280.62,165.19
2017-03,2641,432048.59,163.59
2017-04,2391,412422.24,172.49
2017-05,3660,586190.95,160.16
2017-06,3217,502963.04,156.35
2017-07,3969,584971.62,147.39
2017-08,4293,668204.60,155.65
2017-09,4243,720398.91,169.79
2017-10,4568,769312.37,168.41


Databricks visualization. Run in Databricks to view.

#### Insight 03 — Evolução do ticket médio

A análise demonstrou que o crescimento do faturamento da Olist
esteve associado principalmente ao aumento do volume de pedidos.

Entre janeiro e novembro de 2017, a quantidade mensal de pedidos
passou de 789 para 7.451, representando crescimento de aproximadamente
844%.

No mesmo período, o ticket médio passou de R$ 173,88 para R$ 158,25,
apresentando redução aproximada de 9%.

Esse comportamento indica que o crescimento comercial no intervalo
analisado foi impulsionado principalmente pelo aumento da quantidade
de compras, e não pelo aumento do valor médio dos pedidos.

**Indicador:** ticket médio bruto, incluindo produtos e frete.

### 4. Análise de faturamento por categoria de produto

**Pergunta de negócio:**

Quais categorias de produtos apresentam maior participação
no faturamento do e-commerce?

**Objetivo:**

Identificar as categorias mais relevantes comercialmente,
considerando o faturamento bruto dos produtos vendidos.

O valor do frete será desconsiderado nesta análise para
avaliar especificamente o desempenho comercial dos produtos.

In [0]:
%sql

SELECT
    p.categoria_analitica AS categoria,

    COUNT(*) AS quantidade_itens_vendidos,

    COUNT(DISTINCT f.order_id) AS quantidade_pedidos,

    ROUND(
        SUM(f.valor_produto),
        2
    ) AS faturamento_produtos,

    ROUND(
        SUM(f.valor_produto) * 100.0
        / SUM(SUM(f.valor_produto)) OVER (),
        2
    ) AS participacao_percentual

FROM workspace.gold.fato_itens_pedido f

INNER JOIN workspace.gold.dim_produtos p
    ON f.product_id = p.product_id

GROUP BY
    p.categoria_analitica

ORDER BY
    faturamento_produtos DESC;

categoria,quantidade_itens_vendidos,quantidade_pedidos,faturamento_produtos,participacao_percentual
health_beauty,9670,8836,1258681.34,9.26
watches_gifts,5991,5624,1205005.68,8.87
bed_bath_table,11115,9417,1036988.68,7.63
sports_leisure,8641,7720,988048.97,7.27
computers_accessories,7827,6689,911954.32,6.71
furniture_decor,8334,6449,729762.49,5.37
cool_stuff,3796,3632,635290.85,4.67
housewares,6964,5884,632248.66,4.65
auto,4235,3897,592720.11,4.36
garden_tools,4347,3518,485256.46,3.57


#### Insight 04 — Categorias com maior faturamento

A análise identificou as categorias de produtos com maior
participação no faturamento bruto da Olist.

A categoria de beleza e saúde apresentou o maior faturamento,
com aproximadamente R$ 1,26 milhão, representando 9,26%
do faturamento bruto dos produtos.

Em seguida, destacaram-se relógios e presentes e cama,
mesa e banho.

As cinco categorias com maior faturamento concentraram
aproximadamente 39,74% do valor comercializado.

A comparação entre faturamento e quantidade de itens vendidos
também demonstrou que o maior volume de vendas não implica
necessariamente maior faturamento.

**Observação:** os valores consideram exclusivamente os produtos
vendidos, sem incluir frete.

In [0]:
%sql

SELECT
    p.categoria_analitica AS categoria,

    ROUND(
        SUM(f.valor_produto),
        2
    ) AS faturamento

FROM workspace.gold.fato_itens_pedido f

INNER JOIN workspace.gold.dim_produtos p
    ON f.product_id = p.product_id

GROUP BY
    p.categoria_analitica

ORDER BY
    faturamento DESC

LIMIT 10;

categoria,faturamento
health_beauty,1258681.34
watches_gifts,1205005.68
bed_bath_table,1036988.68
sports_leisure,988048.97
computers_accessories,911954.32
furniture_decor,729762.49
cool_stuff,635290.85
housewares,632248.66
auto,592720.11
garden_tools,485256.46


Databricks visualization. Run in Databricks to view.

### 5. Distribuição geográfica do faturamento

**Pergunta de negócio:**

Quais estados brasileiros apresentam maior participação
no faturamento do e-commerce?

**Objetivo:**

Identificar a concentração geográfica das vendas e
compreender a distribuição regional dos consumidores.

A análise considera o estado de residência do cliente,
não a localização do vendedor.

In [0]:
%sql

SELECT
    c.estado,

    COUNT(DISTINCT p.order_id) AS total_pedidos,

    COUNT(DISTINCT c.customer_unique_id) AS clientes_unicos,

    ROUND(
        SUM(i.valor_produto),
        2
    ) AS faturamento_produtos,

    ROUND(
        SUM(i.valor_frete),
        2
    ) AS valor_frete,

    ROUND(
        SUM(i.valor_total_item),
        2
    ) AS faturamento_total,

    ROUND(
        SUM(i.valor_total_item) * 100.0
        / SUM(SUM(i.valor_total_item)) OVER (),
        2
    ) AS participacao_percentual

FROM workspace.gold.fato_itens_pedido i

INNER JOIN workspace.gold.fato_pedidos p
    ON i.order_id = p.order_id

INNER JOIN workspace.gold.dim_clientes c
    ON p.customer_id = c.customer_id

GROUP BY
    c.estado

ORDER BY faturamento_total DESC
LIMIT 10;

estado,total_pedidos,clientes_unicos,faturamento_produtos,valor_frete,faturamento_total,participacao_percentual
SP,41375,39981,5202955.05,718723.07,5921678.12,37.38
RJ,12762,12303,1824092.67,305589.31,2129681.98,13.44
MG,11544,11178,1585308.03,270853.46,1856161.49,11.72
RS,5432,5249,750304.02,135522.74,885826.76,5.59
PR,4998,4840,683083.76,117851.68,800935.44,5.06
BA,3358,3257,511349.99,100156.68,611506.67,3.86
SC,3612,3513,520553.34,89660.26,610213.60,3.85
DF,2125,2062,302603.94,50625.50,353229.44,2.23
GO,2007,1942,294591.95,53114.98,347706.93,2.19
ES,2025,1956,275037.31,49764.60,324801.91,2.05


Databricks visualization. Run in Databricks to view.

#### Análise dos resultados

A distribuição geográfica das vendas demonstra uma
concentração significativa do faturamento na região Sudeste.

São Paulo representa 37,38% do faturamento total,
seguido pelo Rio de Janeiro (13,44%) e Minas Gerais (11,72%).

Juntos, esses três estados concentram aproximadamente
62,54% do faturamento analisado.

Os resultados evidenciam a importância do mercado
consumidor do Sudeste para as operações comerciais
representadas no dataset da Olist.

### Análise de Logística e Satisfação dos Clientes

Esta análise investiga a relação entre o cumprimento
dos prazos de entrega e a satisfação dos consumidores.

Os dados são obtidos da tabela fato_pedidos da camada Gold,
que integra informações de pedidos, entregas e avaliações.

Objetivo: identificar possíveis diferenças na satisfação
dos clientes entre pedidos entregues no prazo e com atraso.

In [0]:
%sql

SELECT
    status_prazo,

    COUNT(*) AS total_pedidos,

    ROUND(
        AVG(nota_media_avaliacao),
        2
    ) AS nota_media,

    ROUND(
        AVG(tempo_entrega_dias),
        2
    ) AS tempo_medio_entrega,

    ROUND(
        AVG(atraso_dias),
        2
    ) AS atraso_medio_dias

FROM workspace.gold.fato_pedidos

WHERE nota_media_avaliacao IS NOT NULL

GROUP BY status_prazo

ORDER BY total_pedidos DESC;

status_prazo,total_pedidos,nota_media,tempo_medio_entrega,atraso_medio_dias
no_prazo,89448,4.29,11.0,0.0
atrasado,6382,2.27,33.82,10.52
nao_entregue,2843,1.75,null,null


Databricks visualization. Run in Databricks to view.

#### Insights — Logística e Satisfação

A análise evidencia uma associação entre o cumprimento
dos prazos de entrega e a satisfação dos consumidores.

Pedidos entregues no prazo apresentam nota média de 4,29,
enquanto pedidos atrasados registram média de 2,27.

Pedidos não entregues apresentam a menor avaliação,
com nota média de 1,75.

Além disso, pedidos atrasados apresentam tempo médio
de entrega de 33,82 dias, aproximadamente três vezes
o tempo observado nas entregas realizadas no prazo.

Os resultados indicam que o desempenho logístico está
fortemente associado à experiência dos consumidores,
destacando a importância do acompanhamento dos prazos
de entrega nas operações de e-commerce.